# Additional Instructions

Trying to improve LLMGraphTransformer peformance with examples.

In [1]:
import os

import networkx as nx
from langchain.chains import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from lib.llm import LLMGraphTransformer
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import json
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import pandas as pd
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
import chroma
import json_repair

In [2]:

llm_model = "llama3.1:latest"
llm = ChatOllama(
   model=llm_model,
   temperature=0,
   # other params...
)


with open("./data/training_modeling_papers.json", "r") as f:
    train_data = json.load(f)

with open("./data/modeling_papers.json", "r") as f:
    mod_data = json.load(f)
print(len(train_data))
print(len(mod_data))

46
5737


## 1. Review and construct examples

goal - 5-10 examples from train_data, test on mod_data

In [3]:
examples=[]
train_data[0]['abstract']

'Background: Since the appearance of the first case of COVID-19 in Morocco, the cumulative number of reported infectious cases continues to increase and, consequently, the government imposed the containment measure within the country. Our aim is to predict the impact of the compulsory containment on COVID-19 spread. Earlier knowledge of the epidemic characteristics of COVID-19 transmission related to Morocco will be of great interest to establish an optimal plan-of-action to control the epidemic.\n\nMethod: Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values. Furthermore, simulations of different scenarios of containment are performed.\n\nResults: Epidemic characteristics are predicted according to different rates of containment. The basic reproduction number is estimated to be 2.9949, with CI(

In [4]:
def make_example(head,head_type,tail,tail_type,rel):
    return {"text": text,
                 "response": { "head": head, "head_type":head_type,"relation": relationship,
                   "tail": tail, "tail_type": tail_type}}
    

In [5]:
text = ("Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases "
        "in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers "
        "and we estimated the model parameter values.")

head ="susceptible-asymptomatic-infectious model"
head_type="modeling approach"
tail="basic reproduction number"
tail_type="parameter"
relationship="used_to_determine"

examples.append(make_example(head,head_type,tail,tail_type,relationship))
                
tail2 = "control reproduction number"
examples.append(make_example(head,head_type,tail2,tail_type,relationship))

In [6]:
examples

[{'text': 'Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values.',
  'response': {'head': 'susceptible-asymptomatic-infectious model',
   'head_type': 'modeling approach',
   'relation': 'used_to_determine',
   'tail': 'basic reproduction number',
   'tail_type': 'parameter'}},
 {'text': 'Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values.',
  'response': {'head': 'susceptible-asymptomatic-infectious model',
   'head_type': 'modeling approach',
   'relation': 'used_to_determine',
   'tail': 'control reproduction number',
   'tail_type': 'parameter'}}]

In [7]:
text= "The basic reproduction number is estimated to be 2.9949, with CI(2.6729-3.1485)"
head ="basic reproduction number"
head_type="parameter"
tail="2.9949, with CI(2.6729-3.1485)"
tail_type="value"
relationship="parameter_has_value"

examples.append(make_example(head,head_type,tail,tail_type,relationship))

In [8]:
text ="Furthermore, a threshold value of containment rate, below which the epidemic duration is postponed, is determined"
head ="threshold value of containment"
head_type='parameter'
tail='rate below which the epidemic duration is postponed'
tail_type ='parameter'
relationship='parameter_has_definition'
examples.append(make_example(head,head_type,tail,tail_type,relationship))

In [9]:
text="Our findings show that the basic reproduction number reflects a high speed of spread of the epidemic"
head ="basic reproduction number"
head_type="parameter"
tail="high speed of spread of the epidemic"
tail_type="parameter_has_implications"
examples.append(make_example(head,head_type,tail,tail_type,relationship))

In [10]:
text="Furthermore, the compulsory containment can be efficient if more than 73% of population are confined."
head="efficiency of compulsory containment" 
head_type="outcome"
tail="more than 73% of population are confined"
tail_type="threshold"
relationship="outcome_realized_if"
examples.append(make_example(head,head_type,tail,tail_type,relationship))

In [11]:
test ="However, even with 90% of containment, the end-time is estimated to happen on July 4th which can be harmful and lead to consequent social-economic damages."

head="epidemic end time"
head_type="temporal prediction"
tail ="on July 4th"
relationship="is_predicted_to_occur"
examples.append(make_example(head,head_type,tail,tail_type,relationship))

tail2="social-economic damages"
tail2_type="consequences"
relationship="has_implications"
examples.append(make_example(head,head_type,tail2,tail2_type,relationship))

head="90% of containment"
head_type="threshold"
tail="epidemic end time"
tail_type="temporal prediction"
relationship="is necessary_condition"
examples.append(make_example(head,head_type,tail,tail_type,relationship))

In [12]:
test = "sensitivity analysis investigation shows that the COVID-19 dynamics depends strongly on the asymptomatic duration as well as the contact and containment rates"
head1="asymptomatic duration"
head2="contact rates"
head3="containment rates"
tail="COVID-19 dynamics"
head_type="parameter"
tail_type="outcome"
relationship="influences"

examples.append(make_example(head1,head_type,tail,tail_type,relationship))
examples.append(make_example(head2,head_type,tail,tail_type,relationship))
examples.append(make_example(head3,head_type,tail,tail_type,relationship))

## try this with examples 

In [13]:
transformer2 = LLMGraphTransformer(llm=llm,user_examples=examples,user_examples_structured=False)

In [14]:
example_out = [Document(page_content=train_data[0]['abstract'])]
gd = transformer2.convert_to_graph_documents(example_out)
gd

[GraphDocument(nodes=[], relationships=[], source=Document(metadata={}, page_content='Background: Since the appearance of the first case of COVID-19 in Morocco, the cumulative number of reported infectious cases continues to increase and, consequently, the government imposed the containment measure within the country. Our aim is to predict the impact of the compulsory containment on COVID-19 spread. Earlier knowledge of the epidemic characteristics of COVID-19 transmission related to Morocco will be of great interest to establish an optimal plan-of-action to control the epidemic.\n\nMethod: Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values. Furthermore, simulations of different scenarios of containment are performed.\n\nResults: Epidemic characteristics are predicted according to different ra

In [15]:
gd[0].relationships

[]

In [16]:
transformer3 = LLMGraphTransformer(llm=llm,user_examples=examples,user_examples_structured=True)
gd = transformer3.convert_to_graph_documents(example_out)
gd

[GraphDocument(nodes=[Node(id='Covid-19', type='Disease', properties={}), Node(id='Morocco', type='Location', properties={}), Node(id='Government', type='Organization', properties={}), Node(id='Containment Measure', type='Policy', properties={}), Node(id='Basic Reproduction Number', type='Parameter', properties={}), Node(id='Control Reproduction Number', type='Parameter', properties={}), Node(id='Model Parameter Values', type='Data', properties={}), Node(id='Epidemic Duration', type='Outcome', properties={}), Node(id='Asymptomatic Population', type='Population', properties={}), Node(id='Mass Testing', type='Measure', properties={})], relationships=[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Location', properties={}), type='TRANSMITTED_IN', properties={}), Relationship(source=Node(id='Government', type='Organization', properties={}), target=Node(id='Containment Measure', type='Policy', properties={}), type='IMPLEMENTED', prope

In [17]:
gd[0].relationships

[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Location', properties={}), type='TRANSMITTED_IN', properties={}),
 Relationship(source=Node(id='Government', type='Organization', properties={}), target=Node(id='Containment Measure', type='Policy', properties={}), type='IMPLEMENTED', properties={}),
 Relationship(source=Node(id='Basic Reproduction Number', type='Parameter', properties={}), target=Node(id='Covid-19', type='Disease', properties={}), type='CHARACTERISTIC_OF', properties={}),
 Relationship(source=Node(id='Control Reproduction Number', type='Parameter', properties={}), target=Node(id='Covid-19', type='Disease', properties={}), type='CHARACTERISTIC_OF', properties={}),
 Relationship(source=Node(id='Model Parameter Values', type='Data', properties={}), target=Node(id='Basic Reproduction Number', type='Parameter', properties={}), type='ESTIMATED_BY', properties={}),
 Relationship(source=Node(id='Model Parameter Values', ty

## without.

In [18]:
### 4.1 First basic

In [19]:

transformer = LLMGraphTransformer(llm=llm)
graph_documents = transformer.convert_to_graph_documents(example_out)

In [20]:
graph_documents

[GraphDocument(nodes=[Node(id='Covid-19', type='Disease', properties={}), Node(id='Morocco', type='Location', properties={}), Node(id='Government', type='Organization', properties={}), Node(id='Population', type='Group', properties={})], relationships=[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Location', properties={}), type='AFFECTS', properties={}), Relationship(source=Node(id='Government', type='Organization', properties={}), target=Node(id='Population', type='Group', properties={}), type='CONTAINS', properties={}), Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Population', type='Group', properties={}), type='INFECTS', properties={})], source=Document(metadata={}, page_content='Background: Since the appearance of the first case of COVID-19 in Morocco, the cumulative number of reported infectious cases continues to increase and, consequently, the government imposed the containment 

In [21]:
graph_documents[0].nodes

[Node(id='Covid-19', type='Disease', properties={}),
 Node(id='Morocco', type='Location', properties={}),
 Node(id='Government', type='Organization', properties={}),
 Node(id='Population', type='Group', properties={})]

In [22]:
graph_documents[0].relationships

[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Location', properties={}), type='AFFECTS', properties={}),
 Relationship(source=Node(id='Government', type='Organization', properties={}), target=Node(id='Population', type='Group', properties={}), type='CONTAINS', properties={}),
 Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Population', type='Group', properties={}), type='INFECTS', properties={})]

In [23]:
rs =graph_documents[0].relationships

In [24]:
rs

[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Location', properties={}), type='AFFECTS', properties={}),
 Relationship(source=Node(id='Government', type='Organization', properties={}), target=Node(id='Population', type='Group', properties={}), type='CONTAINS', properties={}),
 Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Population', type='Group', properties={}), type='INFECTS', properties={})]

In [25]:
r = rs[0]


In [26]:
r

Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Location', properties={}), type='AFFECTS', properties={})

In [27]:
r.source.id

'Covid-19'

In [28]:
r.source.type

'Disease'

In [ ]:
[ {'source_id':r.source.id,'source_type':r.source.type,'target_id':r.target.id,
   'target_type'=r.target.type,'relationship':r.type} for r in gd1.relationships]